In [ ]:
# docker run -d \
#  -p 8181:8181 \
#  -p 9000:9000 \
#  -p 9001:9001 \
#  --name iceberg-rest \
#  tabulario/iceberg-rest

In [1]:
!pip install "pyiceberg[pyarrow,duckdb]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.6/842.6 kB 10.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 MB 1.6 MB/s  0:00:43 eta 0:00:010:00:010m
  Created wheel for pyiceberg: filename=pyiceberg-0.10.0-cp313-cp313-macosx_26_0_arm64.whl size=623800 sha256=01f8a20c425796f40f8fee786b9a087c9df9f6f6dbda0088b341935945f4541e
  Stored in directory: /Users/jemmeeyung/Library/Caches/pip/wheels/94/e1/ad/72718f6a4b508a4dcd74f62431dc44240ca3518c3837f58600
Successfully built pyiceberg
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pyiceberg]━ 4/5 [pyiceberg]core]


In [4]:
from pyiceberg.catalog import load_catalog
import pandas as pd
import pyarrow as pa  # Standard bridge for schemas

# 1. Connect to the Catalog
catalog = load_catalog(
    "local",
    **{
        "type": "rest",
        "uri": "http://localhost:8181",
        "s3.endpoint": "http://localhost:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
    }
)

# 2. Data from Chronicles
df = pd.DataFrame([
    {"king": "Asa", "reign": 41, "merit": "Good"},
    {"king": "Jehoshaphat", "reign": 25, "merit": "Good"}
])

# 3. FIX: Convert Pandas DataFrame to a PyArrow Table
# This automatically handles the schema "inference" for us
pa_table = pa.Table.from_pandas(df)

# 4. Create the namespace and table
catalog.create_namespace_if_not_exists("chronicles")
identifier = "chronicles.kings_of_judah"

# Clean up previous attempts
try:
    catalog.drop_table(identifier)
except:
    pass

# Use the schema from the Arrow table to create the Iceberg table
table = catalog.create_table(
    identifier,
    schema=pa_table.schema, 
)

# 5. Write the data
table.append(pa_table)

# 6. Verify with DuckDB
con = table.scan().to_duckdb("judah_stats")
print("--- Data from 2 Chronicles ---")
print(con.execute("SELECT * FROM judah_stats").df())

--- Data from 2 Chronicles ---
          king  reign merit
0          Asa     41  Good
1  Jehoshaphat     25  Good
